In [28]:
import pandas as pd
import numpy as np

In [29]:
df=pd.read_csv("all_matches.csv")
df.head()

,date,home_team,away_team,home_score,away_score,tournament,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Scotland,False


In [30]:
pd.unique(df[['home_team','away_team']].values.ravel())

array(['Scotland', 'England', 'Wales', 'Ireland', 'Uruguay', 'Argentina',
       'Austria', 'Hungary', 'Bohemia', 'Belgium', 'France',
       'Switzerland', 'Netherlands', 'British Guiana',
       'Trinidad and Tobago', 'South Africa', 'Germany', 'Sweden',
       'Norway', 'Denmark', 'Italy', 'Chile', 'Finland', 'Luxembourg',
       'Russia', 'Philippines', 'China', 'Brazil', 'Suriname',
       'United States', 'Japan', 'Paraguay', 'Egypt', 'Greece', 'Spain',
       'Czechoslovakia', 'Yugoslavia', 'Estonia', 'Northern Ireland',
       'Costa Rica', 'El Salvador', 'Guatemala', 'Honduras', 'Poland',
       'Portugal', 'Romania', 'New Zealand', 'Australia', 'Latvia',
       'Mexico', 'Lithuania', 'Turkey', 'Aruba', 'Curaçao', 'Bulgaria',
       'Canada', 'Soviet Union', 'Haiti', 'Jamaica', 'Kenya', 'Uganda',
       'Bolivia', 'Azerbaijan', 'Armenia', 'Georgia', 'Peru',
       'British Honduras', 'Dutch East Indies', 'Barbados', 'Nicaragua',
       'Cuba', 'Faroe Islands', 'Iceland', 'Mart

In [31]:
from Confederations import get_confederation

all_teams = pd.unique(df[['home_team', 'away_team']].values.ravel())
unknowns = [t for t in all_teams if get_confederation(t) == "Unknown"]
print(sorted(unknowns))

[]


In [32]:
def get_tournament_weight(tournament, home_team, away_team):
    t = tournament.strip()
    home_conf = get_confederation(home_team)
    away_conf = get_confederation(away_team)
    conf_priority = ["UEFA", "CONMEBOL", "CONCACAF", "CAF", "AFC", "OFC", "Unknown"]
    home_rank = conf_priority.index(home_conf)
    away_rank = conf_priority.index(away_conf)
    top_conf = home_conf if home_rank <= away_rank else away_conf

    # ── Tier 4.0 ──────────────────────────────────────────
    if t == "World Cup":
        return 4.0

    # ── Tier 3.0 ──────────────────────────────────────────
    if t in ["European Championship", "Copa America", "Copa América"]:
        return 3.0

    # ── Tier 2.5 ──────────────────────────────────────────
    if t in ["European Championship qual",
             "Copa America qualifier", "Copa América qualifier",
             "African Nations Cup", "Confederations Cup", "Olympic Games","Euro Ch q & Nordic Ch","Euro Ch q & British Ch","South American Champ","Mundialito"]:
        return 2.5

    if "European Nations League" in t and "CONCACAF" not in t:
        if "A" in t: return 2.5
        elif "B" in t: return 2.0
        elif "C" in t: return 1.75
        elif "D" in t: return 1.5
        else: return 2.5

    # ── Tier 2.0 ──────────────────────────────────────────
    if t == "World Cup qualifier":
        if top_conf in ["UEFA", "CONMEBOL"]: return 2.0
        elif top_conf == "CAF":              return 1.75
        elif top_conf == "CONCACAF":         return 1.75
        elif top_conf == "AFC":              return 1.5
        else:                                return 1.25  # OFC

    if t in ["African Nations Cup qualifier",
             "World Cup q & Nordic Ch", "World Cup q & British Ch",
             "NA Champ & WC qual", "World Cup and CONCACAF Ch q",
             "World Cup and Asian Cup qual", "World Cup and African Cup qual",
             "WC and Oce Cup q", "WC q and Oce Cup"]:
        return 2.0

    # ── Tier 1.75 ─────────────────────────────────────────
    if t in ["CONCACAF Championship", "Asian Cup",
             "CONCACAF Cup", "CONCACAF Series","Intercontinental Champ"]:
        return 1.75

    if "CONCACAF Nations League" in t:
        if "A" in t: return 1.75
        elif "B" in t: return 1.5
        elif "C" in t: return 1.25
        else: return 1.75

    # ── Tier 1.5 ──────────────────────────────────────────
    if t in ["Asian Cup qualifier", "CONCACAF Ch q",
             "Gulf Cup", "Arab Cup",
             "CECAFA Cup", "COSAFA Cup"]:
        return 1.5

    if t in ["Southeast Asian Champ", "South Asian Championship",
             "West Asian Championship", "East Asian Championship",
             "Central American Cup", "CFU Championship",
             "Copa Centenario", "Caribbean Cup",
             "Caribbean Championship","CONCACAF Ch q & Car Ch","CONCACAF Ch q & Car Ch PO","North American Champ","CONCACAF Ch & Car Ch q","Balkan & C European Champ"]:
        return 1.5

    # ── Tier 1.25 ─────────────────────────────────────────
    if t in ["Oceania Nations Cup", "Oceania Nations Cup qualifier",
             "Pacific Games", "Pacific Mini Games",
             "South Pacific Games", "South Pacific Mini Games",
             "Melanesian Cup", "Polynesian Cup"]:
        return 1.25

    # ── Keyword fallbacks ─────────────────────────────────
    t_lower = t.lower()

    if "qualifier" in t_lower or "qual" in t_lower:
        return 1.5

    if any(kw in t_lower for kw in ["championship", "cup", "tournament",
                                     "games", "friendly tournament"]):
        return 1.5

    # ── Tier 1.0 — Friendlies and everything else ─────────
    return 1.0
df['weight'] = df.apply(
    lambda row: get_tournament_weight(row['tournament'], row['home_team'], row['away_team']), 
    axis=1
)

In [33]:
df.groupby('tournament')['weight'].first().sort_values(ascending=False).tail(30)

tournament
CONCACAF Ch q & Car Ch PO        1.50
Friendship Games                 1.50
GANEFO Tournament                1.50
CONCACAF Ch q & Car Ch           1.50
CONCACAF Ch q & C Am Cup         1.50
Gossage Cup                      1.50
Dynasty Cup                      1.50
Pacific Mini Games               1.25
Pacific Games                    1.25
Oceania Nations Cup qualifier    1.25
Oceania Nations Cup              1.25
Melanesian Cup                   1.25
South Pacific Games              1.25
South Pacific Mini Games         1.25
Danube C & King Mihai C          1.00
Windward Islands Champ           1.00
Copa Cent Rev Mayo               1.00
Friendly                         1.00
Copa Circulo de la Prensa        1.00
FIFA Series                      1.00
Jakarta Anniversary Tourn        1.00
Copa Premio Honor Uruguayo       1.00
Artemio Franchi Trophy           1.00
Copa Ciudad de Mexico            1.00
Copa Paz del Chaco               1.00
Trans-Caucasian Champ            1.00
T

In [34]:
import importlib
import elo
importlib.reload(elo)
from elo import update_elo, get_elo, elo_ratings,apply_yearly_decay



In [35]:
df_sorted = df.sort_values('date').reset_index(drop=True)
home_elos = []
away_elos = []
current_year =None

for _, row in df_sorted.iterrows():
    match_year = int(row['date'][:4])
    if current_year is None:
        current_year=match_year
    if match_year>current_year:
        apply_yearly_decay(current_year)
        current_year=match_year
    home_elos.append(get_elo(row['home_team']))
    away_elos.append(get_elo(row['away_team']))
    update_elo(
        row['home_team'], row['away_team'],
        row['home_score'], row['away_score'],
        row['weight'], row['neutral']
    )

df_sorted['home_elo'] = home_elos
df_sorted['away_elo'] = away_elos
df_sorted['elo_diff'] = df_sorted['home_elo'] - df_sorted['away_elo']

if "West Germany" in elo_ratings:
    elo_ratings["Germany"] = max(elo_ratings.get("Germany", 0), elo_ratings["West Germany"])
    del elo_ratings["West Germany"]

if "Soviet Union" in elo_ratings:
    elo_ratings["Russia"] = max(elo_ratings.get("Russia", 0), elo_ratings["Soviet Union"])
    del elo_ratings["Soviet Union"]
    
elo_df = pd.DataFrame(list(elo_ratings.items()), columns=['team', 'elo'])
print(elo_df.sort_values('elo', ascending=False).head(20))

            team          elo
32         Spain  2260.146298
5      Argentina  2144.309846
10        France  2135.139892
1        England  2062.769645
166      Senegal  2044.966348
51        Turkey  2020.198277
87       Ecuador  2017.888530
44      Portugal  2014.338807
30         Japan  2008.829390
27        Brazil  2001.285755
84      Colombia  1993.152275
18        Norway  1989.572030
12   Netherlands  1983.796290
16       Germany  1967.907200
92       Croatia  1948.841294
49        Mexico  1939.651921
11   Switzerland  1920.602411
4        Uruguay  1917.703277
122      Nigeria  1913.742656
31      Paraguay  1911.834666


In [36]:
def get_recent_form(team,past_df,n=10):
    team_matches=past_df[
        (past_df['home_team']==team)|
        (past_df['away_team']==team)
        ].tail(n)
    if len(team_matches)==0:
        return 0.5
    
    results =[]
    for _, match in team_matches.iterrows():
        if match['home_team']==team:
            if match['home_score'] >match['away_score']:results.append(1.0)
            elif match['home_score'] ==match['away_score']:results.append(0.5)
            else: results.append(0.0)
        else:
            if match['home_score'] <match['away_score']:results.append(1.0)
            elif match['home_score'] ==match['away_score']:results.append(0.5)
            else: results.append(0.0)
    weights = np.exp(np.linspace(-2, 0, len(results)))
    return np.average(results, weights=weights)

In [37]:
def get_h2h(home_team,away_team,past_df):
    h2h = past_df[
        ((past_df['home_team'] == home_team) & (past_df['away_team'] == away_team)) |
        ((past_df['home_team'] == away_team) & (past_df['away_team'] == home_team))
    ]
    if len(h2h) == 0:
        return 0.5
    wins = 0
    for _, match in h2h.iterrows():
        if match['home_team'] == home_team:
            if match['home_score'] > match['away_score']: wins += 1
        else:
            if match['away_score'] > match['home_score']:wins+=1
    return wins / len(h2h)

In [38]:
def get_attack_rating(team,past_df,n=10):
    matches=past_df[
        (past_df['home_team'] == team) |
        (past_df['away_team'] == team)
    ].tail(n)
    if len(matches)==0:
        return 1.5
    goals = []
    for _, match in matches.iterrows():
        if match['home_team'] == team:
            goals.append(match['home_score'])
        else:
            goals.append(match['away_score'])
    weights = np.exp(np.linspace(-2, 0, len(goals)))
    return np.average(goals, weights=weights)

def get_defence_rating(team, past_df, n=10):
    matches = past_df[
        (past_df['home_team'] == team) | 
        (past_df['away_team'] == team)
    ].tail(n)
    if len(matches) == 0:
        return 1.5  # league average default
    goals_conceded = []
    for _, match in matches.iterrows():
        if match['home_team'] == team:
            goals_conceded.append(match['away_score'])
        else:
            goals_conceded.append(match['home_score'])
    weights = np.exp(np.linspace(-2, 0, len(goals_conceded)))
    return np.average(goals_conceded, weights=weights)

    

In [39]:
df_check = pd.read_csv('matches_with_features.csv')
print(df_check.shape)
print(df_check.columns.tolist())
print(df_check.isnull().sum())
print(df_check[['attack_home', 'attack_away', 'defence_home', 'defence_away']].head(10))

(51491, 19)
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'country', 'neutral', 'weight', 'home_elo', 'away_elo', 'elo_diff', 'recent_form_home', 'recent_form_away', 'h2h', 'attack_home', 'attack_away', 'defence_home', 'defence_away']
date                0
home_team           0
away_team           0
home_score          0
away_score          0
tournament          0
country             0
neutral             0
weight              0
home_elo            0
away_elo            0
elo_diff            0
recent_form_home    0
recent_form_away    0
h2h                 0
attack_home         0
attack_away         0
defence_home        0
defence_away        0
dtype: int64
   attack_home  attack_away  defence_home  defence_away
0     1.500000     1.500000      1.500000      1.500000
1     0.000000     0.000000      0.000000      0.000000
2     1.761594     3.523188      3.523188      1.761594
3     1.644155     1.819939      1.819939      1.644155
4     1.858462     1.86

In [40]:
df = pd.read_csv('matches_with_features.csv')

df['result']=df.apply(lambda row: 2 if row['home_score']>row['away_score']
                    else(1 if row['home_score']==row['away_score']
                    else 0),axis=1)

df['home_advantage']=df['neutral'].apply(lambda x: 0 if x else 1)

df['attack_diff'] = df['attack_home'] - df['attack_away']
df['defence_diff'] = df['defence_away'] - df['defence_home']
df['form_diff'] = df['recent_form_home'] - df['recent_form_away']


features = ['elo_diff', 'form_diff', 'attack_diff', 'defence_diff',
            'h2h', 'weight', 'home_advantage']


train = df[(df['date'] >= '2000-01-01') & (df['date'] < '2018-01-01')]
test = df[df['date'] >= '2018-01-01']

X_train=train[features]
y_train=train['result']
X_test = test[features]
y_test = test['result']

print(f"Train: {len(X_train)} matches")
print(f"Test: {len(X_test)} matches")

Train: 17170 matches
Test: 8164 matches


In [41]:
def get_team_snapshot(team, df):
    matches = df[
        (df['home_team'] == team) |
        (df['away_team'] == team)
    ].sort_values('date')

    if len(matches) == 0:
        return None

    row = matches.iloc[-1]

    if row['home_team'] == team:
        form = row['recent_form_home']
        attack = row['attack_home']
        defence = row['defence_home']
    else:
        form = row['recent_form_away']
        attack = row['attack_away']
        defence = row['defence_away']

    return {
        'elo': elo_ratings.get(team, 1500),
        'form': form,
        'attack': attack,
        'defence': defence
    }

In [42]:
X_train = train[features]
y_train = train['result']

X_test = test[features]
y_test = test['result']

train = train.copy()

train['sample_weight'] = np.exp(
    (pd.to_datetime(train['date']).rank() / len(train)) * 2
)

In [43]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

In [44]:
# Filter to neutral venue matches only for this model
neutral_train = df[(df['date'] >= '2000-01-01') & 
                   (df['date'] < '2018-01-01') & 
                   (df['neutral'] == True)]

neutral_test = df[(df['date'] >= '2018-01-01') & 
                  (df['neutral'] == True)]

features_neutral = ['elo_diff', 'form_diff', 'attack_diff', 'defence_diff', 'weight']

X_neutral_train = neutral_train[features_neutral]
y_neutral_train = neutral_train['result']
X_neutral_test = neutral_test[features_neutral]
y_neutral_test = neutral_test['result']

print(f"Neutral train: {len(X_neutral_train)}")
print(f"Neutral test: {len(X_neutral_test)}")

Neutral train: 4469
Neutral test: 2643


In [45]:
neutral_train = neutral_train.copy()
neutral_train['sample_weight'] = np.exp(
    (pd.to_datetime(neutral_train['date']).rank() / len(neutral_train)) * 2
)



In [46]:
# Flip every match
neutral_train_flipped = neutral_train.copy()
neutral_train_flipped['elo_diff'] = -neutral_train['elo_diff']
neutral_train_flipped['form_diff'] = -neutral_train['form_diff']
neutral_train_flipped['attack_diff'] = -neutral_train['attack_diff']
neutral_train_flipped['defence_diff'] = -neutral_train['defence_diff']
neutral_train_flipped['result'] = neutral_train['result'].map({2: 0, 0: 2, 1: 1})
neutral_train_flipped['sample_weight'] = neutral_train['sample_weight']


# Combine
neutral_train_aug = pd.concat([neutral_train, neutral_train_flipped]).reset_index(drop=True)

print(neutral_train_aug['result'].value_counts())

# Retrain
X_aug = neutral_train_aug[features_neutral]
y_aug = neutral_train_aug['result']
weights_aug = neutral_train_aug['sample_weight']

xgb_neutral = XGBClassifier(n_estimators=200, max_depth=4,
                             learning_rate=0.05, random_state=42,
                             subsample=0.8, colsample_bytree=0.8)
xgb_neutral.fit(X_aug, y_aug, sample_weight=weights_aug)

# Test
neutral_preds = xgb_neutral.predict(X_neutral_test)
print(f"Neutral accuracy: {accuracy_score(y_neutral_test, neutral_preds):.4f}")

result
2    3394
0    3394
1    2150
Name: count, dtype: int64
Neutral accuracy: 0.5501


In [47]:
def predict_match(team_a, team_b, is_host_a=False, is_host_b=False, weight=4.0):
    a = get_team_snapshot(team_a, df)
    b = get_team_snapshot(team_b, df)
    
    elo_diff = a['elo'] - b['elo']
    form_diff = a['form'] - b['form']
    attack_diff = a['attack'] - b['attack']
    defence_diff = b['defence'] - a['defence']
    
    
    fv = pd.DataFrame([[elo_diff, form_diff, attack_diff, defence_diff, weight]], 
                      columns=features_neutral)
    
    probs = xgb_neutral.predict_proba(fv)[0]
    
    return {
        f'{team_a}_win': round(float(probs[2]), 4),
        'draw': round(float(probs[1]), 4),
        f'{team_b}_win': round(float(probs[0]), 4)
    }

In [48]:
print(predict_match("Spain", "France"))
print(predict_match("France", "Spain"))

print(predict_match("Spain", "Gibraltar"))
print(predict_match("Gibraltar", "Spain"))

{'Spain_win': 0.6776, 'draw': 0.1702, 'France_win': 0.1522}
{'France_win': 0.1459, 'draw': 0.1755, 'Spain_win': 0.6786}
{'Spain_win': 0.9325, 'draw': 0.0311, 'Gibraltar_win': 0.0364}
{'Gibraltar_win': 0.0355, 'draw': 0.0291, 'Spain_win': 0.9354}


In [49]:
print(predict_match("Argentina", "Brazil"))
print(predict_match("Brazil", "Argentina"))

print(predict_match("England", "Germany"))
print(predict_match("Germany", "England"))

{'Argentina_win': 0.4412, 'draw': 0.2421, 'Brazil_win': 0.3167}


{'Brazil_win': 0.2972, 'draw': 0.2593, 'Argentina_win': 0.4435}
{'England_win': 0.6071, 'draw': 0.2299, 'Germany_win': 0.163}
{'Germany_win': 0.1779, 'draw': 0.2425, 'England_win': 0.5796}


In [73]:
def simulate_match(team_a, team_b):
    probs = predict_match(team_a, team_b)

    p_a = probs[f"{team_a}_win"]
    p_draw = probs["draw"]
    p_b = probs[f"{team_b}_win"]

    total = p_a + p_draw + p_b

    p_a /= total
    p_draw /= total
    p_b /= total

    outcome = np.random.choice(
        ["A", "D", "B"],
        p=[p_a, p_draw, p_b]
    )

    return outcome, p_a, p_draw, p_b

In [69]:
def simulate_group_match(team_a, team_b):
    result = simulate_match(team_a, team_b)

    if result == "A":
        return {
            team_a: 3,
            team_b: 0
        }

    elif result == "B":
        return {
            team_a: 0,
            team_b: 3
        }

    else:
        return {
            team_a: 1,
            team_b: 1
        }

In [70]:
def simulate_group(teams):
    points = {team: 0 for team in teams}

    matches = [
        (teams[0], teams[1]),
        (teams[0], teams[2]),
        (teams[0], teams[3]),
        (teams[1], teams[2]),
        (teams[1], teams[3]),
        (teams[2], teams[3])
    ]

    for team_a, team_b in matches:
        result = simulate_group_match(team_a, team_b)

        points[team_a] += result[team_a]
        points[team_b] += result[team_b]

    return sorted(points.items(),
                  key=lambda x: x[1],
                  reverse=True)

In [74]:
print(simulate_match("Spain", "France"))

(np.str_('A'), 0.6776, 0.1702, 0.1522)
